# 011 · Fetch CFPB complaints → landing volume
Pulls complaints from the CFPB API into `/Volumes/<catalog>/bronze/landing/complaints/`,
one newline-delimited JSON file per `date_received` day. Used for both the backfill and daily runs.

### Setup
Imports and the CFPB search API endpoint.

In [0]:
import json, time, requests
from datetime import date, timedelta

API = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"

### API functions
- `get_page`: one page (≤10K rows) for a single day, with retries. API's `date_received_max` is **inclusive**.
- `fetch_day`: all rows for a day via **cursor paging** (`search_after`), since `frm` is capped at 10,000.
  Dedupes on `complaint_id`; returns API total + rows so counts can be checked.

In [0]:
PAGE = 10000

def get_page(day, search_after=None, retries=3):
    params = {"date_received_min": str(day), "date_received_max": str(day),
              "size": PAGE, "no_aggs": "true"}
    if search_after:
        params["search_after"] = f"{search_after[0]}_{search_after[1]}"
    for attempt in range(retries):
        try:
            r = requests.get(API, params=params, timeout=300)
            if r.status_code == 400:
                raise ValueError(f"400 from API: {r.text[:300]}")
            r.raise_for_status()
            return r.json()
        except ValueError:
            raise
        except Exception:
            if attempt == retries - 1:
                raise
            time.sleep(5 * (attempt + 1))

def fetch_day(day):
    page = get_page(day)
    total = page["hits"]["total"]["value"]
    by_id = {}
    while True:
        hits = page["hits"]["hits"]
        if not hits:
            break
        new_ids = {h["_source"]["complaint_id"] for h in hits} - by_id.keys()
        if not new_ids:
            raise RuntimeError(f"{day}: cursor didn't advance (search_after format likely wrong)")
        for h in hits:
            by_id[h["_source"]["complaint_id"]] = h["_source"]
        if len(by_id) >= total or len(hits) < PAGE:
            break
        page = get_page(day, search_after=hits[-1]["sort"])
    return total, list(by_id.values())

#total, rows = fetch_day(date(2026, 10, 1))
#print(f"API total: {total:,} | unique fetched: {len(rows):,} | match: {total == len(rows)}")
#print("dates:", {r['date_received'][:10] for r in rows})

### Parameters
`catalog`; `run_date` (one day; blank = yesterday, used by the daily job);
`start_date`/`end_date` (inclusive, for backfills); `overwrite` (false = skip existing files).
Job parameters override these defaults.

In [0]:
from datetime import date, timedelta

# Input boxes. Job parameters with the same names fill these in automatically.
dbutils.widgets.text("catalog", "cfpb_dev")
dbutils.widgets.text("run_date", "")     # one day; blank = yesterday (what the daily job uses)
dbutils.widgets.text("start_date", "")   # for backfills: fill in BOTH of these
dbutils.widgets.text("end_date", "")
dbutils.widgets.dropdown("overwrite", "false", ["true", "false"])

catalog   = dbutils.widgets.get("catalog")
run_date  = dbutils.widgets.get("run_date").strip()
start_in  = dbutils.widgets.get("start_date").strip()
end_in    = dbutils.widgets.get("end_date").strip()
overwrite = dbutils.widgets.get("overwrite") == "true"

if run_date:                      # one specific day
    start_date = end_date = run_date
elif start_in and end_in:         # a backfill range
    start_date, end_date = start_in, end_in
else:                             # nothing filled in → yesterday
    start_date = end_date = (date.today() - timedelta(days=1)).isoformat()

out_dir = f"/Volumes/{catalog}/bronze/landing/complaints"
dbutils.fs.mkdirs(out_dir)        # make sure the landing folder exists

print(f"Fetching {start_date} → {end_date} into {catalog}")

### Fetch & write
Fetches 4 days in parallel (I/O-bound; kept low for a public API). Skips existing files,
writes via `.tmp` + rename so partial files never land. Reports count mismatches and failures at the end.

In [0]:
import os, json
from datetime import date, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

# Connect to cell 3: turn the text dates into real dates and build the landing path
start   = date.fromisoformat(start_date)
end     = date.fromisoformat(end_date)
out_dir = f"/Volumes/{catalog}/bronze/landing/complaints"

WORKERS = 4

def process_day(d):
    path = f"{out_dir}/complaints_{d}.json"
    if os.path.exists(path) and not overwrite:
        return (str(d), None, None, None, "skipped")
    total, rows = fetch_day(d)
    if not rows:                                   # nothing published yet: don't leave an empty file
        return (str(d), total, 0, 0, "empty")
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")
    os.replace(tmp, path)   # file appears only when complete
    return (str(d), total, len(rows), round(os.path.getsize(path) / 1e6, 1), "ok")

days = [start + timedelta(days=i) for i in range((end - start).days + 1)]
results, failures = [], []

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(process_day, d): d for d in days}
    for fut in as_completed(futures):
        d = futures[fut]
        try:
            res = fut.result()
            results.append(res)
            print(f"{res[0]}: {res[4]}" + (f", {res[2]:,} rows, {res[3]} MB" if res[4] == "ok" else ""))
        except Exception as e:
            failures.append((str(d), str(e)[:200]))
            print(f"{d}: FAILED, {e}")

ok = [r for r in results if r[4] == "ok"]
print(f"\nWrote {len(ok)} days, {sum(r[2] for r in ok):,} rows, {sum(r[3] for r in ok):,.1f} MB")
print("Empty days (not written):", [r[0] for r in results if r[4] == "empty"] or "none")
print("Count mismatches:", [r[:3] for r in ok if r[1] != r[2]] or "none")
print("Failures:", failures or "none")

# Make the job task turn red if any day failed, instead of silently "succeeding"
if failures:
    raise RuntimeError(f"{len(failures)} day(s) failed: {failures}")